In [ ]:
# ============================================================
# CELL 0: EXTRACT THE BEN GRAHAM IMAGES AND MANIFEST
# The input zip holds processed, processed_bg and the manifest.
# We extract only processed_bg and the manifest.
# ============================================================
import os, zipfile

ZIP = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip"
DEST = "/kaggle/working/data"

if not os.path.exists(f"{DEST}/manifest_v2.csv"):
    print("Extracting processed_bg and manifest (this takes a while)...")
    with zipfile.ZipFile(ZIP) as z:
        wanted = [n for n in z.namelist()
                  if n.startswith("processed_bg/") or n == "manifest_v2.csv"]
        print(f"Files to extract: {len(wanted):,}")
        z.extractall(DEST, members=wanted)
    print("Done!")
else:
    print("Already extracted, skipping")

print("Top level:", os.listdir(DEST))

In [ ]:
# ============================================================
# CELL 1: SETUP
# Load the tools, check the GPU is on, and set up logging
# ============================================================
import sys, json, time, random, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score, confusion_matrix,
    f1_score, accuracy_score, cohen_kappa_score,
)
import matplotlib.pyplot as plt

# Without a GPU this would take days, so stop now if it's off
assert torch.cuda.is_available(), (
    "NO GPU. Settings > Accelerator > GPU T4 x2, then restart the session."
)
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True

RUN_NAME = "resnet18_5grade_wce_ls_adamw_steplr_bg"
OUT = "/kaggle/working"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(f"{OUT}/{RUN_NAME}.log")],
    force=True,
)
log = logging.getLogger(RUN_NAME)

log.info("=" * 60)
log.info("RETINASENSE: ResNet-18, 5-grade, weighted CE + label smoothing, AdamW, step LR, Ben Graham")
log.info("=" * 60)
log.info(f"GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}")

In [ ]:
# ============================================================
# CELL 2: SETTINGS
# Every number that could change the result lives here
# ============================================================

BASE = "/kaggle/working/data"
PROC = f"{BASE}/processed_bg"          # Ben Graham preprocessed images
MANIFEST = f"{BASE}/manifest_v2.csv"

SEED = 42
BATCH = 32                             # each image seen once per epoch, no sampler
EPOCHS = 20
PATIENCE = 5                           # stop after 5 epochs with no referable-AUROC gain

# Optimiser and schedule
LR = 3e-4                              # higher start, then step decay
WD = 1e-4                              # AdamW weight decay
STEP_SIZE = 3                          # drop the learning rate every 3 epochs
GAMMA = 0.3                            # multiply the learning rate by 0.3 at each drop

# Regulariser
LABEL_SMOOTH = 0.05                    # 0 becomes 0.05, 1 becomes 0.95

# Five grades: 0 = No DR, 1 = mild, 2 = moderate, 3 = severe, 4 = proliferative
NUM_CLASSES = 5
REFERABLE_FROM_GRADE = 2               # grades 2 to 4 count as referable for the 90% rule
SENS_TARGET = 0.90

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

log.info("SETTINGS")
log.info(f"  Data:            {PROC}")
log.info(f"  Batch / epochs:  {BATCH} / {EPOCHS} (patience {PATIENCE})")
log.info(f"  AdamW lr {LR}, wd {WD}")
log.info(f"  Step decay:      x{GAMMA} every {STEP_SIZE} epochs")
log.info(f"  Label smoothing: {LABEL_SMOOTH}")
log.info(f"  Referable = grade >= {REFERABLE_FROM_GRADE}, sensitivity target {SENS_TARGET:.0%}")

In [ ]:
# ============================================================
# CELL 3: LOAD THE DATA AND COMPUTE THE CLASS WEIGHTS
# Weight for each grade = 1 / sqrt(count), scaled so the average is 1.
# Rare grades get a bigger weight, so each mistake on them costs more.
# ============================================================

df = pd.read_csv(MANIFEST)
ok = df["load_ok"].astype(str).str.lower().eq("true") & df["excluded_reason"].isna()

splits = {}
for s in ["train", "val", "test"]:
    splits[s] = df[(df["split"] == s) & ok].reset_index(drop=True)
    log.info(f"  {s:5s}: {len(splits[s]):,} images")

# Every row needs a grade from 0 to 4
for s, frame in splits.items():
    assert frame["grade"].notna().all(), f"{s} has missing grades"
    grades = set(frame["grade"].astype(int).unique())
    assert grades <= set(range(NUM_CLASSES)), f"{s} has grades outside 0-4: {grades}"

# Count each grade in train
train_counts = np.bincount(splits["train"]["grade"].astype(int), minlength=NUM_CLASSES)
for g in range(NUM_CLASSES):
    log.info(f"  train grade {g}: {train_counts[g]:,} images")

# Class weights: 1 / sqrt(count), then scaled so the average weight is 1
raw_weights = 1.0 / np.sqrt(train_counts)
class_weights = raw_weights / raw_weights.mean()
for g in range(NUM_CLASSES):
    log.info(f"  class weight grade {g}: {class_weights[g]:.2f}")
log.info("  Expected for grades 1 to 4 about 0.80, 0.44, 1.64, 1.12 (differences mean the counts differ)")

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

# Check one image is 512 x 512
first = splits["train"]["filename"].iloc[0]
size = Image.open(f"{PROC}/train/{first}").size
log.info(f"  Image size: {size} (expecting 512 x 512)")
assert size == (512, 512), f"Expected 512x512, got {size}"

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


class RetinaDataset(Dataset):
    """Loads one image and its grade (0 to 4)."""

    def __init__(self, frame, split_name):
        self.paths = [f"{PROC}/{split_name}/{f}" for f in frame["filename"]]
        self.grades = frame["grade"].astype(int).values

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))
        x = torch.from_numpy(img).permute(2, 0, 1)                  # uint8, channels first
        y = torch.tensor(self.grades[i], dtype=torch.long)          # integer grade for CrossEntropy
        return x, y


def make_loader(name, shuffle):
    return DataLoader(
        RetinaDataset(splits[name], name),
        batch_size=BATCH,
        shuffle=shuffle,                      # plain shuffle, no sampler
        num_workers=min(os.cpu_count() or 2, 4),
        pin_memory=True,
        persistent_workers=True,
    )


train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("val", shuffle=False)
test_loader = make_loader("test", shuffle=False)

xb, yb = next(iter(train_loader))
log.info(f"Batch check: images {tuple(xb.shape)} {xb.dtype}, grades {tuple(yb.shape)} {yb.dtype}")
log.info(f"  Batches per epoch: {len(train_loader)}")

In [ ]:
# ============================================================
# CELL 4: THE MODEL
# ResNet-18 with ImageNet weights and a new 5-output head
# ============================================================

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)   # 512 in, 5 grade scores out
model = model.to(DEVICE)

for p in model.parameters():
    p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
log.info(f"Model: ResNet-18 (ImageNet weights), 5 outputs")
log.info(f"  Trainable parameters: {n_train:,} of {n_total:,}")
assert n_train == n_total, "Something is frozen, but this run trains all layers"

In [ ]:
# ============================================================
# CELL 5: LOSS, OPTIMISER, SCHEDULE
# Weighted cross-entropy with label smoothing (training only).
# The validation loss uses the same weights but no smoothing.
# ============================================================

criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=LABEL_SMOOTH)
val_criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
scaler = torch.amp.GradScaler("cuda")

log.info(f"Loss: weighted CE, label smoothing {LABEL_SMOOTH}")
log.info(f"Optimiser: AdamW lr {LR}, wd {WD} | schedule: StepLR x{GAMMA} every {STEP_SIZE} epochs")
log.info("Mixed precision: ON")

In [ ]:
# ============================================================
# CELL 6: TRAINING AND VALIDATION
# Train up to 20 epochs. After each one, score the validation set
# and keep the epoch with the best referable AUROC.
# ============================================================


def to_input(x):
    """Scale 0-255 pixels to 0-1, then apply ImageNet normalisation."""
    x = x.to(DEVICE, non_blocking=True).float().div_(255.0)
    return (x - MEAN) / STD


@torch.no_grad()
def predict(loader):
    """Return softmax probabilities for the 5 grades, the true grades, and the average loss."""
    model.eval()
    probs_all, labels_all, total, n = [], [], 0.0, 0
    for x, y in loader:
        y = y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(to_input(x))
        logits = logits.float()
        total += val_criterion(logits, y).item() * len(y)
        n += len(y)
        probs_all.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels_all.append(y.cpu().numpy())
    return np.concatenate(probs_all), np.concatenate(labels_all), total / n


def grade_metrics(labels, probs):
    """Five-grade metrics plus the referable (grade 2 and up) AUROC used for early stopping."""
    preds = probs.argmax(axis=1)
    p_ref = probs[:, REFERABLE_FROM_GRADE:].sum(axis=1)
    ref_labels = (labels >= REFERABLE_FROM_GRADE).astype(int)
    metrics = {
        "acc5": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro")),
        "qwk": float(cohen_kappa_score(labels, preds, weights="quadratic")),
        "ref_auroc": float(roc_auc_score(ref_labels, p_ref)),
    }
    return metrics, p_ref


history = []
best_auc, bad_epochs = 0.0, 0

log.info("=" * 60)
log.info("TRAINING STARTED")
log.info("=" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    run_loss, seen = 0.0, 0

    for step, (x, y) in enumerate(train_loader, 1):
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(to_input(x))
        loss = criterion(logits.float(), y)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        run_loss += loss.item() * len(y)
        seen += len(y)
        if step == 1 or step % 200 == 0:
            log.info(f"  epoch {epoch} | batch {step}/{len(train_loader)} | "
                     f"running loss {run_loss / seen:.4f}")

    train_loss = run_loss / seen
    scheduler.step()                              # step decay, once per epoch
    lr_now = optimizer.param_groups[0]["lr"]

    probs, labels, val_loss = predict(val_loader)
    metrics, _ = grade_metrics(labels, probs)
    minutes = (time.time() - t0) / 60

    improved = metrics["ref_auroc"] > best_auc
    if improved:
        best_auc, bad_epochs = metrics["ref_auroc"], 0
        torch.save(model.state_dict(), f"{OUT}/{RUN_NAME}_best.pt")
        status = "NEW BEST, saved"
    else:
        bad_epochs += 1
        status = f"no improvement ({bad_epochs}/{PATIENCE})"

    history.append({
        "epoch": epoch, "train_loss": round(train_loss, 5), "val_loss": round(val_loss, 5),
        "val_ref_auroc": round(metrics["ref_auroc"], 5), "val_acc5": round(metrics["acc5"], 5),
        "val_macro_f1": round(metrics["macro_f1"], 5), "val_qwk": round(metrics["qwk"], 5),
        "lr_after_epoch": lr_now, "minutes": round(minutes, 2),
    })

    log.info("-" * 60)
    log.info(f"EPOCH {epoch} COMPLETE in {minutes:.1f} min | lr now {lr_now:.2e}")
    log.info(f"  train loss {train_loss:.4f} | val loss {val_loss:.4f}")
    log.info(f"  val referable AUROC {metrics['ref_auroc']:.4f} | 5-grade acc {metrics['acc5']:.4f} | "
             f"macro F1 {metrics['macro_f1']:.4f} | QWK {metrics['qwk']:.4f}")
    log.info(f"  best referable AUROC {best_auc:.4f} | {status}")
    log.info("-" * 60)

    if bad_epochs >= PATIENCE:
        log.info("EARLY STOPPING: referable AUROC stopped improving")
        break

log.info(f"TRAINING DONE: {len(history)} epochs, best val referable AUROC {best_auc:.4f}")

with open(f"{OUT}/{RUN_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ============================================================
# CELL 7: CHOOSE THE THRESHOLD
# Reload the best model. Use p(referable) = grade 2 + grade 3 + grade 4.
# Find the first cutoff that catches 90% of referable eyes on validation.
# ============================================================

model.load_state_dict(torch.load(f"{OUT}/{RUN_NAME}_best.pt", map_location=DEVICE))
log.info("Loaded best checkpoint")

val_probs, val_labels, _ = predict(val_loader)
val_ref_labels = (val_labels >= REFERABLE_FROM_GRADE).astype(int)
val_p_ref = val_probs[:, REFERABLE_FROM_GRADE:].sum(axis=1)

fpr, tpr, thr = roc_curve(val_ref_labels, val_p_ref)
i = np.where(tpr >= SENS_TARGET)[0][0]
threshold = float(thr[i])

log.info(f"THRESHOLD {threshold:.4f} | val sensitivity {tpr[i]:.3f} | "
         f"val specificity {1 - fpr[i]:.3f} | FROZEN for test")

In [ ]:
# ============================================================
# CELL 8: FINAL TEST
# Score the test set once with the frozen threshold.
# DO NOT run this cell more than once.
# ============================================================

test_probs, test_labels, test_loss = predict(test_loader)
test_metrics, test_p_ref = grade_metrics(test_labels, test_probs)

# Five-grade confusion matrix and per-grade recall
test_preds5 = test_probs.argmax(axis=1)
cm5 = confusion_matrix(test_labels, test_preds5, labels=list(range(NUM_CLASSES)))
per_grade_recall = {}
for g in range(NUM_CLASSES):
    total_g = cm5[g].sum()
    per_grade_recall[f"grade_{g}"] = float(cm5[g, g] / total_g) if total_g > 0 else 0.0

# Referable (grade 2 and up) at the frozen threshold
ref_labels = (test_labels >= REFERABLE_FROM_GRADE).astype(int)
ref_pred = (test_p_ref >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(ref_labels, ref_pred, labels=[0, 1]).ravel()

results = {
    "run": RUN_NAME,
    "model": "resnet18",
    "task": "5-grade, referable = grade >= 2",
    "loss": "weighted_ce_label_smoothing",
    "optimiser": "adamw_steplr",
    "preprocessing": "ben_graham",
    "class_weights": [float(w) for w in class_weights],
    "test_5grade": {
        "accuracy": test_metrics["acc5"],
        "macro_f1": test_metrics["macro_f1"],
        "quadratic_kappa": test_metrics["qwk"],
        "per_grade_recall": per_grade_recall,
        "confusion_5x5": cm5.tolist(),
    },
    "test_referable": {
        "threshold": threshold,
        "sensitivity": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "precision": float(tp / (tp + fp)) if (tp + fp) else 0.0,
        "auroc": float(roc_auc_score(ref_labels, test_p_ref)),
        "pr_auc": float(average_precision_score(ref_labels, test_p_ref)),
        "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    },
}

log.info("RESULTS")
log.info(f"  5-grade accuracy {results['test_5grade']['accuracy']:.4f} | "
         f"macro F1 {results['test_5grade']['macro_f1']:.4f} | "
         f"QWK {results['test_5grade']['quadratic_kappa']:.4f}")
for g in range(NUM_CLASSES):
    log.info(f"  recall grade {g}: {per_grade_recall[f'grade_{g}']:.4f}")
log.info(f"  referable sensitivity {results['test_referable']['sensitivity']:.4f} | "
         f"specificity {results['test_referable']['specificity']:.4f} | "
         f"AUROC {results['test_referable']['auroc']:.4f} | "
         f"PR-AUC {results['test_referable']['pr_auc']:.4f}")
log.info(f"  confusion (referable): TN={tn} FP={fp} FN={fn} TP={tp}")

with open(f"{OUT}/{RUN_NAME}_results.json", "w") as f:
    json.dump({"results": results, "history": history}, f, indent=2)
log.info("DONE. Do not rerun this cell.")

In [ ]:
# ============================================================
# CELL 9: CHARTS
# ============================================================
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

ax[0].plot(h["epoch"], h["train_loss"], marker="o", label="train loss")
ax[0].plot(h["epoch"], h["val_loss"], marker="s", label="val loss")
ax[0].set_title("Weighted CE loss")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(h["epoch"], h["val_ref_auroc"], marker="o", color="green")
ax[1].set_title("Validation referable AUROC (grade 2+)")
ax[1].set_xlabel("Epoch")

ax[2].plot(h["epoch"], h["val_qwk"], marker="o", label="quadratic kappa")
ax[2].plot(h["epoch"], h["val_macro_f1"], marker="s", label="macro F1")
ax[2].set_title("Validation 5-grade metrics")
ax[2].set_xlabel("Epoch")
ax[2].legend()

plt.tight_layout()
plt.savefig(f"{OUT}/{RUN_NAME}_curves.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(cm5, cmap="Blues")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm5[i, j]), ha="center", va="center",
                color="white" if cm5[i, j] > cm5.max() / 2 else "black")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels([f"pred {g}" for g in range(NUM_CLASSES)])
ax.set_yticklabels([f"true {g}" for g in range(NUM_CLASSES)])
ax.set_title("Test 5-grade confusion matrix")
plt.tight_layout()
plt.savefig(f"{OUT}/{RUN_NAME}_confusion5.png", dpi=150)
plt.show()
log.info("Charts saved")